In [ ]:
import time

import feedparser
import pandas as pd
from newspaper import Article

In [ ]:
# the SAP news feed only shows the latest ~30 posts on page 1,
# so we page back a few times to get more volume
base_url = "https://news.sap.com/feed/"

sap_docs = []

for page in range(1, 6):
    feed_url = base_url if page == 1 else f"{base_url}?paged={page}"
    feed = feedparser.parse(feed_url)

    if not feed.entries:
        break

    for entry in feed.entries:
        sap_docs.append({
            "title": entry.title,
            "url": entry.link,
            "published": entry.published,
            "source": "SAP News",
            "description": ""
        })

    time.sleep(0.3)

len(sap_docs)

In [ ]:
sap_df = pd.DataFrame(sap_docs).drop_duplicates(subset="url").reset_index(drop=True)

len(sap_df)

In [ ]:
# scrape the full article text for each entry, skip the ones that fail
contents = []

for url in sap_df["url"]:

    try:

        article = Article(url)

        article.download()
        article.parse()

        contents.append(article.text)

    except:

        contents.append("")

In [ ]:
sap_df["content"] = contents

In [ ]:
sap_df.to_json(
    "../data/sap_news.json",
    orient="records",
    indent=2
)

print("saved", len(sap_df), "sap articles")